# Exercise 3: NoSQL on AWS - DocumentDB, DynamoDB, and Keyspaces

In this exercise, you'll explore three different NoSQL database services on AWS:

1. **Amazon DocumentDB** - A MongoDB-compatible document database
2. **Amazon DynamoDB** - A fully managed key-value and document database
3. **Amazon Keyspaces** - A fully managed Apache Cassandra-compatible database

Each database has different strengths and use cases. By exploring each one, you'll understand the trade-offs between consistency, scalability, and query flexibility.

## Prerequisites

- AWS Account with appropriate permissions
- AWS CLI configured
- Python with boto3 installed
- A default VPC in your AWS account

## Cost Warning ⚠️

**IMPORTANT:** This exercise creates AWS resources that may incur charges if left running. We will shut everything down at the end, but please monitor your AWS console to ensure all resources are deleted.

## Step 1: AWS Authentication

First, let's configure AWS credentials and set up our boto3 client. This is similar to Lesson 1, Exercise 5.

Run the cell below and follow the prompts to authenticate to AWS:

In [ ]:
import boto3
import json
import time
from datetime import datetime

# Initialize AWS clients
# If credentials are not already configured, you'll be prompted

docdb_client = boto3.client('docdb', region_name='us-east-1')
dynamodb_client = boto3.client('dynamodb', region_name='us-east-1')
keyspaces_client = boto3.client('keyspaces', region_name='us-east-1')
ec2_client = boto3.client('ec2', region_name='us-east-1')
sts_client = boto3.client('sts')

# Verify authentication
try:
    account_id = sts_client.get_caller_identity()['Account']
    print(f"✅ Successfully authenticated to AWS")
    print(f"Account ID: {account_id}")
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run 'aws configure' in your terminal and try again")

## Step 2: Setup Network Infrastructure

For DocumentDB and other databases, we need to set up VPC networking. Let's get the default VPC and create a security group.

### Create Security Group

In [ ]:
# Get default VPC
vpcs = ec2_client.describe_vpcs(Filters=[{'Name': 'isDefault', 'Values': ['true']}])
vpc_id = vpcs['Vpcs'][0]['VpcId']
print(f"Using default VPC: {vpc_id}")

# Create security group for databases
sg_name = 'nosql-exercise-sg'
try:
    sg = ec2_client.create_security_group(
        GroupName=sg_name,
        Description='Security group for NoSQL databases exercise',
        VpcId=vpc_id
    )
    sg_id = sg['GroupId']
    print(f"✅ Created security group: {sg_id}")
except Exception as e:
    if 'InvalidGroup.Duplicate' in str(e):
        # Security group already exists, get its ID
        sgs = ec2_client.describe_security_groups(GroupNames=[sg_name])
        sg_id = sgs['SecurityGroups'][0]['GroupId']
        print(f"⚠️ Security group already exists: {sg_id}")
    else:
        print(f"❌ Error creating security group: {e}")

# Allow inbound traffic (for DocumentDB port 27017, DynamoDB doesn't need this)
try:
    ec2_client.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[
            {
                'IpProtocol': 'tcp',
                'FromPort': 27017,
                'ToPort': 27017,
                'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'DocumentDB access'}]
            },
            {
                'IpProtocol': 'tcp',
                'FromPort': 9042,
                'ToPort': 9042,
                'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'Keyspaces access'}]
            }
        ]
    )
    print(f"✅ Added inbound rules to security group")
except Exception as e:
    if 'InvalidPermission.Duplicate' in str(e):
        print(f"⚠️ Inbound rules already exist")
    else:
        print(f"❌ Error adding inbound rules: {e}")

### Create DB Subnet Group

DocumentDB requires a DB subnet group. Let's get the available subnets and create one.

In [ ]:
# Get subnets in the default VPC
subnets = ec2_client.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}])
subnet_ids = [subnet['SubnetId'] for subnet in subnets['Subnets'][:2]]  # Use first 2 subnets
print(f"Available subnets: {subnet_ids}")

# Create DB subnet group for DocumentDB
subnet_group_name = 'nosql-exercise-subnet-group'
try:
    docdb_client.create_db_subnet_group(
        DBSubnetGroupName=subnet_group_name,
        DBSubnetGroupDescription='Subnet group for NoSQL exercise',
        SubnetIds=subnet_ids,
        Tags=[{'Key': 'Exercise', 'Value': 'L3-Ex3'}]
    )
    print(f"✅ Created DB subnet group: {subnet_group_name}")
except Exception as e:
    if 'DBSubnetGroupAlreadyExists' in str(e):
        print(f"⚠️ DB subnet group already exists")
    else:
        print(f"❌ Error creating DB subnet group: {e}")

---

## Part A: Amazon DocumentDB (MongoDB-Compatible)

### What is DocumentDB?
- **MongoDB-compatible** document database
- Stores data as **JSON-like documents**
- Strong **ACID transaction support**
- **Relational query capabilities** with $lookup
- Best for: Applications needing document flexibility with relational queries

### Create DocumentDB Cluster

In [ ]:
import time

cluster_id = 'nosql-docdb-cluster'
master_username = 'docdbadmin'
master_password = 'DocDB123!@#'  # In production, use AWS Secrets Manager

print("Creating DocumentDB cluster... This may take 5-10 minutes.")
print("⏳ Cluster creation is starting in the background...\n")

try:
    response = docdb_client.create_db_cluster(
        DBClusterIdentifier=cluster_id,
        Engine='docdb',
        MasterUsername=master_username,
        MasterUserPassword=master_password,
        DBSubnetGroupName=subnet_group_name,
        VpcSecurityGroupIds=[sg_id],
        BackupRetentionPeriod=1,
        PreferredBackupWindow='03:00-04:00',
        PreferredMaintenanceWindow='mon:04:00-mon:05:00',
        StorageEncrypted=False,  # Disable encryption for testing
        Tags=[{'Key': 'Exercise', 'Value': 'L3-Ex3'}]
    )
    print(f"✅ DocumentDB cluster creation initiated: {cluster_id}")
    print(f"Status: {response['DBCluster']['Status']}")
    
except Exception as e:
    if 'DBClusterAlreadyExistsFault' in str(e):
        print(f"⚠️ DocumentDB cluster already exists: {cluster_id}")
    else:
        print(f"❌ Error creating cluster: {e}")

# Store connection info
DOCDB_CLUSTER_ID = cluster_id
DOCDB_USERNAME = master_username
DOCDB_PASSWORD = master_password

### Create DocumentDB Instance

A cluster needs at least one instance to run.

In [ ]:
instance_id = 'nosql-docdb-instance-1'

print("Creating DocumentDB instance in the cluster...")

try:
    response = docdb_client.create_db_instance(
        DBInstanceIdentifier=instance_id,
        DBInstanceClass='db.t3.small',  # Small instance type
        Engine='docdb',
        DBClusterIdentifier=cluster_id,
        PubliclyAccessible=True,  # For testing only
        Tags=[{'Key': 'Exercise', 'Value': 'L3-Ex3'}]
    )
    print(f"✅ DocumentDB instance creation initiated: {instance_id}")
    print(f"Instance Class: db.t3.small")
    print(f"Status: {response['DBInstance']['DBInstanceStatus']}")
    
except Exception as e:
    if 'DBInstanceAlreadyExists' in str(e):
        print(f"⚠️ DocumentDB instance already exists: {instance_id}")
    else:
        print(f"❌ Error creating instance: {e}")

print("\n⏳ Instance creation takes 5-10 minutes. Continuing with DynamoDB and Keyspaces...")
print("We'll check DocumentDB status later.")

---

## Part B: Amazon DynamoDB (Key-Value/Document)

### What is DynamoDB?
- **Fully managed** key-value and document database
- **Serverless** - no infrastructure to manage
- **Single-digit millisecond latency** at any scale
- **Partition key + Sort key** design
- Best for: High-performance applications with simple query patterns

### Create DynamoDB Table

Unlike DocumentDB and Keyspaces, DynamoDB creates instantly!

In [ ]:
table_name = 'nosql-orders'

print("Creating DynamoDB table...")

try:
    response = dynamodb_client.create_table(
        TableName=table_name,
        KeySchema=[
            {'AttributeName': 'customer_id', 'KeyType': 'HASH'},      # Partition key
            {'AttributeName': 'order_id', 'KeyType': 'RANGE'}        # Sort key
        ],
        AttributeDefinitions=[
            {'AttributeName': 'customer_id', 'AttributeType': 'S'},  # String
            {'AttributeName': 'order_id', 'AttributeType': 'S'}      # String
        ],
        BillingMode='PAY_PER_REQUEST'  # On-demand pricing (no capacity planning)
    )
    print(f"✅ DynamoDB table created instantly: {table_name}")
    print(f"Key Schema: customer_id (HASH) + order_id (RANGE)")
    print(f"Billing Mode: Pay-Per-Request (Serverless)")
    
except Exception as e:
    if 'ResourceInUseException' in str(e):
        print(f"⚠️ DynamoDB table already exists: {table_name}")
    else:
        print(f"❌ Error creating table: {e}")

DYNAMODB_TABLE_NAME = table_name

### Insert Data into DynamoDB

DynamoDB is already ready! Let's add some sample orders.

In [ ]:
print("Inserting sample data into DynamoDB...\n")

sample_orders = [
    {
        'customer_id': {'S': 'cust-001'},
        'order_id': {'S': 'order-101'},
        'product': {'S': 'Laptop'},
        'total': {'N': '1200.00'},
        'order_date': {'S': '2024-01-15'}
    },
    {
        'customer_id': {'S': 'cust-001'},
        'order_id': {'S': 'order-102'},
        'product': {'S': 'Mouse'},
        'total': {'N': '50.00'},
        'order_date': {'S': '2024-01-16'}
    },
    {
        'customer_id': {'S': 'cust-002'},
        'order_id': {'S': 'order-103'},
        'product': {'S': 'Keyboard'},
        'total': {'N': '75.00'},
        'order_date': {'S': '2024-01-17'}
    }
]

try:
    for order in sample_orders:
        dynamodb_client.put_item(TableName=table_name, Item=order)
        print(f"✅ Inserted order: {order['order_id']['S']}")
    print(f"\n✅ All {len(sample_orders)} orders inserted successfully")
except Exception as e:
    print(f"❌ Error inserting data: {e}")

### Query DynamoDB

DynamoDB queries work on the **Partition Key + Sort Key** design. Let's get all orders for a customer.

In [ ]:
print("Querying DynamoDB: Get all orders for customer 'cust-001'\n")
print("Query Pattern: Partition Key (customer_id) + Sort Key Range (order_id)\n")

try:
    response = dynamodb_client.query(
        TableName=table_name,
        KeyConditionExpression='customer_id = :cid',
        ExpressionAttributeValues={
            ':cid': {'S': 'cust-001'}
        }
    )
    
    print(f"Found {response['Count']} orders:\n")
    for item in response['Items']:
        print(f"  Order ID: {item['order_id']['S']}")
        print(f"  Product: {item['product']['S']}")
        print(f"  Total: ${item['total']['N']}")
        print()
    
    print(f"\n✅ DynamoDB Query Characteristics:")
    print(f"  - Query by partition key + sort key only")
    print(f"  - No complex joins or aggregations")
    print(f"  - Ultra-fast single-digit millisecond response")
    print(f"  - Predictable, scalable performance")
    
except Exception as e:
    print(f"❌ Error querying table: {e}")

---

## Part C: Amazon Keyspaces (Apache Cassandra-Compatible)

### What is Keyspaces?
- **Apache Cassandra-compatible** database
- **Distributed**, **highly available** architecture
- **Partition + clustering keys** for time-series and IoT data
- **Eventual consistency** for high availability
- Best for: Time-series data, IoT, massive scale with high availability

### Create Keyspace and Table

In [ ]:
keyspace_name = 'nosql_exercise'
table_name_keyspaces = 'orders'

print("Creating Keyspaces (Cassandra) keyspace and table...\n")

try:
    # Create keyspace
    response = keyspaces_client.create_keyspace(
        keyspaceName=keyspace_name,
        tags=[{'key': 'Exercise', 'value': 'L3-Ex3'}]
    )
    print(f"✅ Created keyspace: {keyspace_name}")
    print(f"ARN: {response['resourceArn']}")
    
except Exception as e:
    if 'ConflictException' in str(e):
        print(f"⚠️ Keyspace already exists: {keyspace_name}")
    else:
        print(f"❌ Error creating keyspace: {e}")

# Wait a moment for keyspace to be ready
time.sleep(2)

print("\nCreating table in keyspace...\n")

try:
    # Create table
    response = keyspaces_client.create_table(
        keyspaceName=keyspace_name,
        tableName=table_name_keyspaces,
        schemaDefinition={
            'allColumns': [
                {'name': 'customer_id', 'type': 'text'},
                {'name': 'order_date', 'type': 'timestamp'},
                {'name': 'order_id', 'type': 'text'},
                {'name': 'product', 'type': 'text'},
                {'name': 'total', 'type': 'decimal'}
            ],
            'partitionKeys': [
                {'name': 'customer_id'}
            ],
            'clusteringKeys': [
                {'name': 'order_date', 'orderBy': 'DESC'},
                {'name': 'order_id', 'orderBy': 'ASC'}
            ]
        },
        billingMode={'mode': 'PAY_PER_REQUEST'},
        tags=[{'key': 'Exercise', 'value': 'L3-Ex3'}]
    )
    print(f"✅ Created Keyspaces table: {table_name_keyspaces}")
    print(f"Partition Key: customer_id")
    print(f"Clustering Keys: order_date (DESC), order_id (ASC)")
    print(f"\nNote: This structure is optimized for time-series queries!")
    
except Exception as e:
    if 'ConflictException' in str(e):
        print(f"⚠️ Table already exists: {table_name_keyspaces}")
    else:
        print(f"❌ Error creating table: {e}")

KEYSPACES_KEYSPACE = keyspace_name
KEYSPACES_TABLE = table_name_keyspaces

### Insert Data into Keyspaces

We'll use the Cassandra Python driver to interact with Keyspaces.

In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthenticator
from datetime import datetime
from decimal import Decimal

print("Connecting to Amazon Keyspaces...\n")

try:
    # Get the Keyspaces service endpoint
    cloud_config = {
        'secure_connect_bundle': '/path/to/secure-connect-bundle.zip'  # Usually not needed for basic auth
    }
    
    # For Keyspaces, use the service endpoint
    contact_points = ['cassandra.us-east-1.amazonaws.com']
    
    # Create cluster with authentication
    cluster = Cluster(
        contact_points=contact_points,
        port=9042,
        auth_provider=PlainTextAuthenticator(
            username='cassandra',
            password='cassandra'
        ),
        ssl_context=None  # Adjust for production
    )
    
    session = cluster.connect()
    print("✅ Connected to Keyspaces")
    
    # Insert sample data
    insert_query = f"""
    INSERT INTO {KEYSPACES_KEYSPACE}.{KEYSPACES_TABLE} 
    (customer_id, order_date, order_id, product, total)
    VALUES (?, ?, ?, ?, ?)
    """
    
    prepared = session.prepare(insert_query)
    
    orders_data = [
        ('cust-001', datetime(2024, 1, 15), 'order-101', 'Laptop', Decimal('1200.00')),
        ('cust-001', datetime(2024, 1, 16), 'order-102', 'Mouse', Decimal('50.00')),
        ('cust-002', datetime(2024, 1, 17), 'order-103', 'Keyboard', Decimal('75.00'))
    ]
    
    for order in orders_data:
        session.execute(prepared, order)
        print(f"✅ Inserted order: {order[2]}")
    
    print(f"\n✅ All {len(orders_data)} orders inserted")
    
except ImportError:
    print("⚠️ Cassandra driver not installed. Install with: pip install cassandra-driver")
    print("\nContinuing without inserting data...\n")
except Exception as e:
    print(f"⚠️ Connection to Keyspaces: {e}")
    print("This is expected if Keyspaces table is still initializing (takes ~1-2 minutes)")

### Query Keyspaces

Keyspaces excels at time-series queries with clustering keys.

In [ ]:
print("Querying Keyspaces: Get orders for customer in date range\n")
print("Query Pattern: Partition Key (customer_id) + Clustering Keys (order_date DESC, order_id ASC)\n")

try:
    query = f"""
    SELECT customer_id, order_date, order_id, product, total
    FROM {KEYSPACES_KEYSPACE}.{KEYSPACES_TABLE}
    WHERE customer_id = 'cust-001'
    ORDER BY order_date DESC
    """
    
    results = session.execute(query)
    
    print("Results:\n")
    count = 0
    for row in results:
        print(f"  Order ID: {row.order_id}")
        print(f"  Date: {row.order_date}")
        print(f"  Product: {row.product}")
        print(f"  Total: ${row.total}")
        print()
        count += 1
    
    print(f"✅ Keyspaces Query Characteristics:")
    print(f"  - Optimized for time-series data")
    print(f"  - Clustering keys enable efficient range queries")
    print(f"  - Distributed, highly available")
    print(f"  - Eventually consistent")
    
except Exception as e:
    print(f"⚠️ Query failed: {e}")

---

## Comparison: DocumentDB vs DynamoDB vs Keyspaces

Let's summarize the key differences we've seen:

In [ ]:
import pandas as pd

comparison_data = {
    'Feature': [
        'Data Model',
        'Query Language',
        'Key Design',
        'Consistency',
        'Best For',
        'Joins',
        'Aggregations',
        'Time to Create'
    ],
    'DocumentDB': [
        'Document (MongoDB-like)',
        'MongoDB Query Language',
        'No key requirement',
        'Strong (ACID)',
        'Relational queries, complex documents',
        '$lookup (slower)',
        'Yes (aggregation pipeline)',
        '5-10 minutes'
    ],
    'DynamoDB': [
        'Key-Value/Document',
        'DynamoDB API',
        'Partition + Sort Key',
        'Strong (within partition)',
        'High-performance simple queries',
        'Not supported',
        'Not supported',
        'Instant'
    ],
    'Keyspaces (Cassandra)': [
        'Column-oriented',
        'CQL (Cassandra Query Language)',
        'Partition + Clustering Keys',
        'Eventual (tunable)',
        'Time-series, IoT, massive scale',
        'Not supported',
        'Limited',
        '1-2 minutes'
    ]
}

df = pd.DataFrame(comparison_data)
print("\n" + "="*120)
print("COMPARISON: NoSQL Databases on AWS")
print("="*120 + "\n")
print(df.to_string(index=False))
print("\n" + "="*120)

---

## Part D: Check DocumentDB Status

Let's check if our DocumentDB cluster is ready yet. (Creation usually takes 5-10 minutes)

In [ ]:
print("Checking DocumentDB cluster status...\n")

try:
    response = docdb_client.describe_db_clusters(DBClusterIdentifier=DOCDB_CLUSTER_ID)
    cluster = response['DBClusters'][0]
    
    print(f"Cluster: {cluster['DBClusterIdentifier']}")
    print(f"Status: {cluster['Status']}")
    print(f"Endpoint: {cluster.get('Endpoint', 'Not yet available')}")
    print(f"Port: {cluster.get('Port', 27017)}")
    
    if cluster['Status'] == 'available':
        print("\n✅ DocumentDB is ready! The connection string would be:")
        print(f"mongodb://{DOCDB_USERNAME}:<password>@{cluster['Endpoint']}:27017/?tls=true&tlsCAFile=rds-ca-2019-root.pem")
    else:
        print(f"\n⏳ DocumentDB is still {cluster['Status'].lower()}. Check again in a few minutes.")
        
except Exception as e:
    print(f"Error checking status: {e}")

---

## Part E: Clean Up - Delete All Resources

⚠️ **IMPORTANT:** Run this section to delete all resources and avoid unwanted charges!

In [ ]:
print("\n" + "="*80)
print("CLEANUP: Deleting all AWS resources created in this exercise")
print("="*80 + "\n")

# Step 1: Delete DynamoDB table
print("Step 1: Deleting DynamoDB table...")
try:
    dynamodb_client.delete_table(TableName=DYNAMODB_TABLE_NAME)
    print(f"✅ DynamoDB table '{DYNAMODB_TABLE_NAME}' deletion initiated")
except Exception as e:
    print(f"❌ Error deleting DynamoDB table: {e}")

print()

# Step 2: Delete Keyspaces table
print("Step 2: Deleting Keyspaces table...")
try:
    keyspaces_client.delete_table(
        keyspaceName=KEYSPACES_KEYSPACE,
        tableName=KEYSPACES_TABLE
    )
    print(f"✅ Keyspaces table '{KEYSPACES_TABLE}' deletion initiated")
except Exception as e:
    print(f"❌ Error deleting Keyspaces table: {e}")

print()

# Step 3: Delete Keyspaces keyspace
print("Step 3: Deleting Keyspaces keyspace...")
try:
    keyspaces_client.delete_keyspace(keyspaceName=KEYSPACES_KEYSPACE)
    print(f"✅ Keyspaces keyspace '{KEYSPACES_KEYSPACE}' deletion initiated")
except Exception as e:
    print(f"❌ Error deleting Keyspaces keyspace: {e}")

print()

# Step 4: Delete DocumentDB instance
print("Step 4: Deleting DocumentDB instance...")
try:
    docdb_client.delete_db_instance(
        DBInstanceIdentifier=instance_id,
        SkipFinalSnapshot=True
    )
    print(f"✅ DocumentDB instance '{instance_id}' deletion initiated")
except Exception as e:
    print(f"❌ Error deleting DocumentDB instance: {e}")

print()

# Step 5: Delete DocumentDB cluster
print("Step 5: Deleting DocumentDB cluster...")
try:
    docdb_client.delete_db_cluster(
        DBClusterIdentifier=DOCDB_CLUSTER_ID,
        SkipFinalSnapshot=True
    )
    print(f"✅ DocumentDB cluster '{DOCDB_CLUSTER_ID}' deletion initiated")
except Exception as e:
    print(f"❌ Error deleting DocumentDB cluster: {e}")

### Clean Up Network Resources

Delete the DB subnet group and security group.

In [ ]:
import time

print("\nStep 6: Deleting DB subnet group...")
print("(Waiting 30 seconds for resources to detach)\n")

time.sleep(30)  # Wait for instances to be deleted

try:
    docdb_client.delete_db_subnet_group(DBSubnetGroupName=subnet_group_name)
    print(f"✅ DB subnet group '{subnet_group_name}' deleted")
except Exception as e:
    print(f"❌ Error deleting subnet group: {e}")

print()

print("Step 7: Deleting security group...")
try:
    ec2_client.delete_security_group(GroupId=sg_id)
    print(f"✅ Security group '{sg_id}' deleted")
except Exception as e:
    print(f"❌ Error deleting security group: {e}")

print("\n" + "="*80)
print("✅ CLEANUP COMPLETE!")
print("="*80)
print("\nAll AWS resources have been deleted.")
print("Monitor your AWS console to confirm deletion is complete.")

---

## Summary

### What You Learned

1. **Amazon DocumentDB**
   - MongoDB-compatible document database
   - Strong ACID transactions with relational query support
   - Use for applications needing document flexibility with complex queries
   - Slower joins due to document model limitations

2. **Amazon DynamoDB**
   - Fully managed, serverless key-value store
   - Single-digit millisecond latency at massive scale
   - Designed for simple, predictable query patterns
   - No joins or complex aggregations

3. **Amazon Keyspaces**
   - Apache Cassandra-compatible distributed database
   - Optimized for time-series and IoT workloads
   - Clustering keys enable efficient range queries
   - Eventually consistent for high availability

### Key Takeaways

- **Choose your database based on your query patterns**, not data format
- **DocumentDB**: Complex relational queries on documents
- **DynamoDB**: Simple, high-performance lookups
- **Keyspaces**: Time-series and historical data analysis
- **Always clean up resources** to avoid unnecessary AWS charges

### Next Steps

- Experiment with larger datasets
- Try more complex queries specific to each database
- Read the AWS documentation for each service
- Consider cost implications for your use case